In [321]:
from sklearn.pipeline import Pipeline
import joblib
import os
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

In [206]:
import numpy as np
import pandas as pd
tr = pd.read_csv('train_FD001.txt', sep=r'\s+', header=None)

In [207]:
from ydata_profiling import ProfileReport

In [208]:
%pip install -q ydata-profiling

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [209]:
import pandas as pd
tr = pd.read_csv('train_FD001.txt', sep=r'\s+', header=None)

In [210]:
tr.head()

,0,1,2,3,4,5,6,7,8,9,...,16,17,18,19,20,21,22,23,24,25
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044


In [211]:
tr.tail()

,0,1,2,3,4,5,6,7,8,9,...,16,17,18,19,20,21,22,23,24,25
20626,100,196,-0.0004,-0.0003,100.0,518.67,643.49,1597.98,1428.63,14.62,...,519.49,2388.26,8137.60,8.4956,0.03,397,2388,100.0,38.49,22.9735
20627,100,197,-0.0016,-0.0005,100.0,518.67,643.54,1604.50,1433.58,14.62,...,519.68,2388.22,8136.50,8.5139,0.03,395,2388,100.0,38.30,23.1594
20628,100,198,0.0004,0.0000,100.0,518.67,643.42,1602.46,1428.18,14.62,...,520.01,2388.24,8141.05,8.5646,0.03,398,2388,100.0,38.44,22.9333
20629,100,199,-0.0011,0.0003,100.0,518.67,643.23,1605.26,1426.53,14.62,...,519.67,2388.23,8139.29,8.5389,0.03,395,2388,100.0,38.29,23.0640
20630,100,200,-0.0032,-0.0005,100.0,518.67,643.85,1600.38,1432.14,14.62,...,519.30,2388.26,8137.33,8.5036,0.03,396,2388,100.0,38.37,23.0522


In [212]:
tr.shape

(20631, 26)

In [213]:
tr.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20631 entries, 0 to 20630
Data columns (total 26 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   0       20631 non-null  int64  
 1   1       20631 non-null  int64  
 2   2       20631 non-null  float64
 3   3       20631 non-null  float64
 4   4       20631 non-null  float64
 5   5       20631 non-null  float64
 6   6       20631 non-null  float64
 7   7       20631 non-null  float64
 8   8       20631 non-null  float64
 9   9       20631 non-null  float64
 10  10      20631 non-null  float64
 11  11      20631 non-null  float64
 12  12      20631 non-null  float64
 13  13      20631 non-null  float64
 14  14      20631 non-null  float64
 15  15      20631 non-null  float64
 16  16      20631 non-null  float64
 17  17      20631 non-null  float64
 18  18      20631 non-null  float64
 19  19      20631 non-null  float64
 20  20      20631 non-null  float64
 21  21      20631 non-null  int64  
 22

In [214]:
tr.describe()

,0,1,2,3,4,5,6,7,8,9,...,16,17,18,19,20,21,22,23,24,25
count,20631.000000,20631.000000,20631.000000,20631.000000,20631.0,20631.00,20631.000000,20631.000000,20631.000000,2.063100e+04,...,20631.000000,20631.000000,20631.000000,20631.000000,2.063100e+04,20631.000000,20631.0,20631.0,20631.000000,20631.000000
mean,51.506568,108.807862,-0.000009,0.000002,100.0,518.67,642.680934,1590.523119,1408.933782,1.462000e+01,...,521.413470,2388.096152,8143.752722,8.442146,3.000000e-02,393.210654,2388.0,100.0,38.816271,23.289705
std,29.227633,68.880990,0.002187,0.000293,0.0,0.00,0.500053,6.131150,9.000605,5.329200e-15,...,0.737553,0.071919,19.076176,0.037505,3.469531e-18,1.548763,0.0,0.0,0.180746,0.108251
min,1.000000,1.000000,-0.008700,-0.000600,100.0,518.67,641.210000,1571.040000,1382.250000,1.462000e+01,...,518.690000,2387.880000,8099.940000,8.324900,3.000000e-02,388.000000,2388.0,100.0,38.140000,22.894200
25%,26.000000,52.000000,-0.001500,-0.000200,100.0,518.67,642.325000,1586.260000,1402.360000,1.462000e+01,...,520.960000,2388.040000,8133.245000,8.414900,3.000000e-02,392.000000,2388.0,100.0,38.700000,23.221800
50%,52.000000,104.000000,0.000000,0.000000,100.0,518.67,642.640000,1590.100000,1408.040000,1.462000e+01,...,521.480000,2388.090000,8140.540000,8.438900,3.000000e-02,393.000000,2388.0,100.0,38.830000,23.297900
75%,77.000000,156.000000,0.001500,0.000300,100.0,518.67,643.000000,1594.380000,1414.555000,1.462000e+01,...,521.950000,2388.140000,8148.310000,8.465600,3.000000e-02,394.000000,2388.0,100.0,38.950000,23.366800
max,100.000000,362.000000,0.008700,0.000600,100.0,518.67,644.530000,1616.910000,1441.490000,1.462000e+01,...,523.380000,2388.560000,8293.720000,8.584800,3.000000e-02,400.000000,2388.0,100.0,39.430000,23.618400


In [215]:
tr.sample(5)

,0,1,2,3,4,5,6,7,8,9,...,16,17,18,19,20,21,22,23,24,25
11236,56,257,0.0002,-0.0000,100.0,518.67,643.58,1599.84,1419.84,14.62,...,520.31,2388.14,8195.68,8.5419,0.03,396,2388,100.0,38.43,23.0820
17091,84,207,0.0009,0.0004,100.0,518.67,642.95,1601.07,1418.06,14.62,...,520.72,2388.16,8131.64,8.4654,0.03,394,2388,100.0,38.48,23.1700
4138,20,205,0.0008,-0.0003,100.0,518.67,642.60,1594.30,1425.26,14.62,...,520.23,2388.23,8142.39,8.4833,0.03,395,2388,100.0,38.60,23.1094
13543,68,112,0.0030,-0.0004,100.0,518.67,642.92,1589.10,1407.88,14.62,...,521.55,2388.11,8150.92,8.4388,0.03,395,2388,100.0,38.68,23.2873
15236,76,78,-0.0043,-0.0004,100.0,518.67,642.67,1590.02,1418.20,14.62,...,521.18,2388.06,8130.81,8.4470,0.03,395,2388,100.0,38.77,23.3318


In [216]:
#######################################################################

In [217]:
columns = [
    "unit_id",
    "cycle",
    "setting_1",
    "setting_2",
    "setting_3",
    "sensor_1",
    "sensor_2",
    "sensor_3",
    "sensor_4",
    "sensor_5",
    "sensor_6",
    "sensor_7",
    "sensor_8",
    "sensor_9",
    "sensor_10",
    "sensor_11",
    "sensor_12",
    "sensor_13",
    "sensor_14",
    "sensor_15",
    "sensor_16",
    "sensor_17",
    "sensor_18",
    "sensor_19",
    "sensor_20",
    "sensor_21"
]

tr.columns = columns

In [218]:
tr.head()

,unit_id,cycle,setting_1,setting_2,setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044


In [219]:
tr.shape

(20631, 26)

In [220]:
###################################################################################

In [221]:
#cleaning data
tr_clean = tr.copy()
tr_clean.dtypes

unit_id        int64
cycle          int64
setting_1    float64
setting_2    float64
setting_3    float64
sensor_1     float64
sensor_2     float64
sensor_3     float64
sensor_4     float64
sensor_5     float64
sensor_6     float64
sensor_7     float64
sensor_8     float64
sensor_9     float64
sensor_10    float64
sensor_11    float64
sensor_12    float64
sensor_13    float64
sensor_14    float64
sensor_15    float64
sensor_16    float64
sensor_17      int64
sensor_18      int64
sensor_19    float64
sensor_20    float64
sensor_21    float64
dtype: object

In [222]:
tr_clean.isnull().sum()

unit_id      0
cycle        0
setting_1    0
setting_2    0
setting_3    0
sensor_1     0
sensor_2     0
sensor_3     0
sensor_4     0
sensor_5     0
sensor_6     0
sensor_7     0
sensor_8     0
sensor_9     0
sensor_10    0
sensor_11    0
sensor_12    0
sensor_13    0
sensor_14    0
sensor_15    0
sensor_16    0
sensor_17    0
sensor_18    0
sensor_19    0
sensor_20    0
sensor_21    0
dtype: int64

In [223]:
tr_clean.duplicated().sum()

np.int64(0)

In [224]:
constant_columns = []
for col in tr_clean.columns:
    unique_count = tr_clean[col].nunique()
    if unique_count <= 1:
        constant_columns.append(col)
print(constant_columns)

['setting_3', 'sensor_1', 'sensor_5', 'sensor_10', 'sensor_16', 'sensor_18', 'sensor_19']


In [225]:
negative_values = (tr_clean < 0).sum()

negative_values

unit_id          0
cycle            0
setting_1    10061
setting_2     9225
setting_3        0
sensor_1         0
sensor_2         0
sensor_3         0
sensor_4         0
sensor_5         0
sensor_6         0
sensor_7         0
sensor_8         0
sensor_9         0
sensor_10        0
sensor_11        0
sensor_12        0
sensor_13        0
sensor_14        0
sensor_15        0
sensor_16        0
sensor_17        0
sensor_18        0
sensor_19        0
sensor_20        0
sensor_21        0
dtype: int64

In [226]:
tr_clean = tr_clean.drop(columns=constant_columns)

In [227]:
tr_clean = tr_clean.sort_values(
    by=["unit_id", "cycle"]
).reset_index(drop=True)

In [228]:
print(tr_clean.shape)
print(tr_clean.columns.tolist())

(20631, 19)
['unit_id', 'cycle', 'setting_1', 'setting_2', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_6', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_17', 'sensor_20', 'sensor_21']


In [229]:
print("Missing values:", tr_clean.isnull().sum().sum())
print("Duplicate rows:", tr_clean.duplicated().sum())

Missing values: 0
Duplicate rows: 0


In [230]:
####################################################################

In [231]:
engine_summary = tr_clean.groupby("unit_id")["cycle"].max().reset_index()

engine_summary.head(10)

,unit_id,cycle
0,1,192
1,2,287
2,3,179
3,4,189
4,5,269
5,6,188
6,7,259
7,8,150
8,9,201
9,10,222


In [232]:
print("Number of engines:", engine_summary["unit_id"].nunique())
engine_summary = engine_summary.rename(columns={"cycle": "total_cycles"})
print("Maximum engine cycles:", engine_summary["total_cycles"].max())
print("Average engine cycles:", engine_summary["total_cycles"].mean())

Number of engines: 100
Maximum engine cycles: 362
Average engine cycles: 206.31


In [233]:
all_sorted = True

for engine_id in tr_clean["unit_id"].unique():
    engine_cycles = tr_clean[
        tr_clean["unit_id"] == engine_id
    ]["cycle"]

    if not engine_cycles.is_monotonic_increasing:
        all_sorted = False
        print("Engine", engine_id, "is not sorted")

print("All engines sorted:", all_sorted)

All engines sorted: True


In [234]:
engine_summary["total_cycles"].describe()

count    100.000000
mean     206.310000
std       46.342749
min      128.000000
25%      177.000000
50%      199.000000
75%      229.250000
max      362.000000
Name: total_cycles, dtype: float64

In [235]:
############################################################

In [236]:
tr_clean["RUL"] = (
    tr_clean.groupby("unit_id")["cycle"].transform("max")
    - tr_clean["cycle"]
)

tr_clean[["unit_id", "cycle", "RUL"]].head()

,unit_id,cycle,RUL
0,1,1,191
1,1,2,190
2,1,3,189
3,1,4,188
4,1,5,187


In [237]:
tr_clean[tr_clean["unit_id"] == 1][
    ["unit_id", "cycle", "RUL"]
].tail()

,unit_id,cycle,RUL
187,1,188,4
188,1,189,3
189,1,190,2
190,1,191,1
191,1,192,0


In [238]:
print("Minimum RUL:", tr_clean["RUL"].min())
print("Maximum RUL:", tr_clean["RUL"].max())
print("Missing RUL values:", tr_clean["RUL"].isnull().sum())

Minimum RUL: 0
Maximum RUL: 361
Missing RUL values: 0


In [239]:
tr_clean["RUL_capped"] = tr_clean["RUL"].clip(upper=125)

In [240]:
tr_clean[["unit_id", "cycle", "RUL", "RUL_capped"]].head()

,unit_id,cycle,RUL,RUL_capped
0,1,1,191,125
1,1,2,190,125
2,1,3,189,125
3,1,4,188,125
4,1,5,187,125


In [241]:
print("Maximum original RUL:", tr_clean["RUL"].max())
print("Maximum capped RUL:", tr_clean["RUL_capped"].max())
print("Rows affected by capping:", (tr_clean["RUL"] > 125).sum())

Maximum original RUL: 361
Maximum capped RUL: 125
Rows affected by capping: 8031


In [242]:
######################################################################
#Features 

In [243]:
columns_to_drop = [
    "unit_id",
    "RUL",
    "RUL_capped"
]

X = tr_clean.drop(columns=columns_to_drop)
y = tr_clean["RUL_capped"]

In [244]:
print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("Feature columns:")
print(X.columns.tolist())

Features shape: (20631, 18)
Target shape: (20631,)
Feature columns:
['cycle', 'setting_1', 'setting_2', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_6', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_17', 'sensor_20', 'sensor_21']


In [245]:
print("Missing values in X:", X.isnull().sum().sum())
print("Missing values in y:", y.isnull().sum())
print("Data types:")
print(X.dtypes)

Missing values in X: 0
Missing values in y: 0
Data types:
cycle          int64
setting_1    float64
setting_2    float64
sensor_2     float64
sensor_3     float64
sensor_4     float64
sensor_6     float64
sensor_7     float64
sensor_8     float64
sensor_9     float64
sensor_11    float64
sensor_12    float64
sensor_13    float64
sensor_14    float64
sensor_15    float64
sensor_17      int64
sensor_20    float64
sensor_21    float64
dtype: object


In [246]:
######Baseline Target & Feature

In [247]:
X_raw = X.copy()

In [248]:
X_engineered = X_raw.copy()

X_engineered["cycle_squared"] = X_engineered["cycle"] ** 2

In [249]:
print(X_engineered.shape)
print(X_engineered.head())

(20631, 19)
   cycle  setting_1  setting_2  sensor_2  sensor_3  sensor_4  sensor_6  \
0      1    -0.0007    -0.0004    641.82   1589.70   1400.60     21.61   
1      2     0.0019    -0.0003    642.15   1591.82   1403.14     21.61   
2      3    -0.0043     0.0003    642.35   1587.99   1404.20     21.61   
3      4     0.0007     0.0000    642.35   1582.79   1401.87     21.61   
4      5    -0.0019    -0.0002    642.37   1582.85   1406.22     21.61   

   sensor_7  sensor_8  sensor_9  sensor_11  sensor_12  sensor_13  sensor_14  \
0    554.36   2388.06   9046.19      47.47     521.66    2388.02    8138.62   
1    553.75   2388.04   9044.07      47.49     522.28    2388.07    8131.49   
2    554.26   2388.08   9052.94      47.27     522.42    2388.03    8133.23   
3    554.45   2388.11   9049.48      47.13     522.86    2388.08    8133.83   
4    554.00   2388.06   9055.15      47.28     522.19    2388.04    8133.80   

   sensor_15  sensor_17  sensor_20  sensor_21  cycle_squared  
0    

In [250]:
##########Data Leakage

In [251]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [252]:
from sklearn.model_selection import GroupShuffleSplit

In [253]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_indices, validation_indices = next(
    splitter.split(
        X,
        y,
        groups=tr_clean["unit_id"]
    )
)

In [254]:
X_train = X.iloc[train_indices]
X_validation = X.iloc[validation_indices]

y_train = y.iloc[train_indices]
y_validation = y.iloc[validation_indices]

groups_train = tr_clean["unit_id"].iloc[train_indices]
groups_validation = tr_clean["unit_id"].iloc[validation_indices]

In [255]:
print("X_train shape:", X_train.shape)
print("X_validation shape:", X_validation.shape)

print("Training engines:", groups_train.nunique())
print("Validation engines:", groups_validation.nunique())

X_train shape: (16561, 18)
X_validation shape: (4070, 18)
Training engines: 80
Validation engines: 20


In [256]:
#################Feature Engineering

In [257]:
X_engineered = X.copy()
X_engineered["cycle_squared"] = X_engineered["cycle"] ** 2

sensor_columns = [
    col for col in X.columns
    if col.startswith("sensor_")
]

for sensor in sensor_columns:

    rolling_values = (
        tr_clean.groupby("unit_id")[sensor]
        .rolling(window=5, min_periods=1)
        .mean()
        .reset_index(level=0, drop=True)
        .sort_index()
    )

    X_engineered[f"{sensor}_rolling_mean"] = rolling_values.values

In [258]:
for sensor in sensor_columns:

    rolling_std_values = (
        tr_clean.groupby("unit_id")[sensor]
        .rolling(
            window=5,
            min_periods=2
        )
        .std()
        .reset_index(level=0, drop=True)
        .sort_index()
    )

    X_engineered[f"{sensor}_rolling_std"] = (
        rolling_std_values
        .fillna(0)
        .values
    )
    

In [259]:
std_columns = [
    col for col in X_engineered.columns
    if col.endswith("_rolling_std")
]
print("Number of rolling std features:", len(std_columns))

Number of rolling std features: 15


In [260]:
print("Original features:", X.shape)
print("Engineered features:", X_engineered.shape)
print("Missing values:", X_engineered.isnull().sum().sum())

Original features: (20631, 18)
Engineered features: (20631, 49)
Missing values: 0


In [261]:
X_train = X_engineered.iloc[train_indices]
X_validation = X_engineered.iloc[validation_indices]

print("X_train:", X_train.shape)
print("X_validation:", X_validation.shape)

X_train: (16561, 49)
X_validation: (4070, 49)


In [262]:
####Model Linear Regression

In [263]:
linear_model = LinearRegression()

linear_model.fit(X_train, y_train)

y_pred = linear_model.predict(X_validation)

In [264]:
mae = mean_absolute_error(y_validation, y_pred)
rmse = np.sqrt(mean_squared_error(y_validation, y_pred))
r2 = r2_score(y_validation, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2)

MAE: 14.179876455275874
RMSE: 17.492684107065067
R2: 0.8241612019155976


In [265]:
#####Random Forest

In [266]:
rf_model = RandomForestRegressor(
    n_estimators=200,
    min_samples_leaf=2,
    max_features=0.8,
    random_state=42,
)

In [267]:
rf_model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",2
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",0.8
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease o

In [268]:
y_pred_rf = rf_model.predict(X_validation)

In [269]:
rf_mae = mean_absolute_error(y_validation, y_pred_rf)

rf_rmse = np.sqrt(
    mean_squared_error(y_validation, y_pred_rf)
)

rf_r2 = r2_score(y_validation, y_pred_rf)

print("Random Forest Results")
print("MAE:", rf_mae)
print("RMSE:", rf_rmse)
print("R2:", rf_r2)

Random Forest Results
MAE: 10.035323932230751
RMSE: 15.157265685220104
R2: 0.8679788455529779


In [270]:
print("Comparison")
print("Linear Regression MAE:", mae)
print("Random Forest MAE:", rf_mae)

print("Linear Regression RMSE:", rmse)
print("Random Forest RMSE:", rf_rmse)

print("Linear Regression R2:", r2)
print("Random Forest R2:", rf_r2)

Comparison
Linear Regression MAE: 14.179876455275874
Random Forest MAE: 10.035323932230751
Linear Regression RMSE: 17.492684107065067
Random Forest RMSE: 15.157265685220104
Linear Regression R2: 0.8241612019155976
Random Forest R2: 0.8679788455529779


In [271]:
################### DecisionTree

In [272]:
dt_model = DecisionTreeRegressor(
    max_depth=15,
    min_samples_leaf=3,
    random_state=42
)

In [273]:
dt_model.fit(X_train, y_train)

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.For an example of how ``max_depth`` influences the model, see:ref:`sphx_glr_auto_examples_tree_plot_tree_regression.py`.",15
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",3
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 0.24 Poisson deviance criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",None
,"max_leaf_nodes ma

In [274]:
y_pred_dt = dt_model.predict(X_validation)

In [275]:
dt_mae = mean_absolute_error(
    y_validation,
    y_pred_dt
)

dt_rmse = np.sqrt(
    mean_squared_error(
        y_validation,
        y_pred_dt
    )
)

dt_r2 = r2_score(
    y_validation,
    y_pred_dt
)

print("Decision Tree Results")
print("MAE:", dt_mae)
print("RMSE:", dt_rmse)
print("R2:", dt_r2)

Decision Tree Results
MAE: 12.989694663385986
RMSE: 20.548327015907724
R2: 0.7573642797925884


In [276]:
####model svm

In [277]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_validation_scaled = scaler.transform(X_validation)

svr_model = SVR()

svr_model.fit(X_train_scaled, y_train)

svr_predictions = svr_model.predict(X_validation_scaled)

In [278]:
svr_mae = mean_absolute_error(
    y_validation,
    svr_predictions
)

svr_rmse = np.sqrt(
    mean_squared_error(
        y_validation,
        svr_predictions
    )
)

svr_r2 = r2_score(
    y_validation,
    svr_predictions
)

print("SVR Results")
print("MAE:", svr_mae)
print("RMSE:", svr_rmse)
print("R2:", svr_r2)

SVR Results
MAE: 11.339444322751827
RMSE: 15.57272002718054
R2: 0.8606423702461845


In [279]:
######model Knn

In [280]:
knn_model = KNeighborsRegressor(
    n_neighbors=10,
    weights="distance",
    p=2,
)

In [281]:
knn_model.fit(X_train_scaled, y_train)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",10
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Uniform weights are used by default.See the following example for a demonstration of the impact ofdifferent weighting schemes on predictions::ref:`sphx_glr_auto_examples_neighbors_plot_regression.py`.",'distance'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric metric: str, DistanceMetric object or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.If metric is a DistanceMetric object, it will be passed directly tothe underlying computation routines.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.Doesn't affect :meth:`fit` method.",None
Name,Type,Value
"effective_metric_ effective_metric_: str or callableThe distance metric to use. It will be same as the `metric` parameteror a synonym of it, e.g. 'euclidean' if the `metric` parameter set to'minkowski' and `p` parameter set to 2.",str,'eu...an'
"effective_metric_params_ effective_metric_params_: dictAdditional keyword arguments for the metric function. For most metricswill be same with `metric_params` parameter, but may also contain the`p` parameter value if the `effective_metric_` attribute is set to'minkowski'.",dict,{}


In [282]:
knn_predictions = knn_model.predict(X_validation_scaled)

In [283]:
knn_mae = mean_absolute_error(
    y_validation,
    knn_predictions
)

knn_rmse = np.sqrt(
    mean_squared_error(
        y_validation,
        knn_predictions
    )
)

knn_r2 = r2_score(
    y_validation,
    knn_predictions
)

print("KNN Results")
print("MAE:", knn_mae)
print("RMSE:", knn_rmse)
print("R2:", knn_r2)

KNN Results
MAE: 11.557504730175063
RMSE: 16.22133372481845
R2: 0.8487919481465065


In [284]:
#########ـBagging Model

In [285]:
bagging_model = BaggingRegressor(
    estimator=DecisionTreeRegressor(
        max_depth=15,
        min_samples_leaf=3,
        random_state=42
    ),
    n_estimators=100,
    max_samples=0.8,
    bootstrap=True,
    random_state=42
)

In [286]:
bagging_model.fit(X_train, y_train)

,"estimator estimator: object, default=NoneThe base estimator to fit on random subsets of the dataset.If None, then the base estimator is a:class:`~sklearn.tree.DecisionTreeRegressor`... versionadded:: 1.2 `base_estimator` was renamed to `estimator`.",DecisionTreeR...ndom_state=42)
,"n_estimators n_estimators: int, default=10The number of base estimators in the ensemble.",100
,"max_samples max_samples: int or float, default=NoneThe number of samples to draw from X to train each base estimator (withreplacement by default, see `bootstrap` for more details).- If None, then draw `X.shape[0]` samples irrespective of `sample_weight`.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` unweighted samples or `max_samples * sample_weight.sum()` weighted samples.",0.8
,"random_state random_state: int, RandomState instance or None, default=NoneControls the random resampling of the original dataset(sample wise and feature wise).If the base estimator accepts a `random_state` attribute, a differentseed is generated for each instance in the ensemble.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator (without replacement by default, see `bootstrap_features` for moredetails).- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.",1.0
,"bootstrap bootstrap: bool, default=TrueWhether samples are drawn with replacement. If False, sampling withoutreplacement is performed. If fitting with `sample_weight`, it isstrongly recommended to choose True, as only drawing with replacementwill ensure the expected frequency semantics of `sample_weight`.",True
,"bootstrap_features bootstrap_features: bool, default=FalseWhether features are drawn with replacement.",False
,"oob_score oob_score: bool, default=FalseWhether to use out-of-bag samples to estimatethe generalization error. Only available if bootstrap=True.",False
,"warm_start warm_start: bool, default=FalseWhen set to True, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fita whole new ensemble. See :term:`the Glossary <warm_start>`.",False
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for both :meth:`fit` and:meth:`predict`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"verbose verbose: int, default=0Controls the verbosity when fitting and predicting.",0


In [287]:
bagging_predictions = bagging_model.predict(X_validation)

In [288]:
bagging_mae = mean_absolute_error(
    y_validation,
    bagging_predictions
)

bagging_rmse = np.sqrt(
    mean_squared_error(
        y_validation,
        bagging_predictions
    )
)

bagging_r2 = r2_score(
    y_validation,
    bagging_predictions
)

print("Bagging Results")
print("MAE:", bagging_mae)
print("RMSE:", bagging_rmse)
print("R2:", bagging_r2)

Bagging Results
MAE: 9.998103212844777
RMSE: 15.13917671932589
R2: 0.8682937705786836


In [289]:
######AdaBoost with Decision Tree

In [290]:
adaboost_model = AdaBoostRegressor(
    estimator=DecisionTreeRegressor(
        max_depth=3,
        min_samples_leaf=3,
        random_state=42
    ),
    n_estimators=150,
    learning_rate=0.05,
    loss="square",
    random_state=42
)

In [291]:
adaboost_model.fit(X_train, y_train)

,"estimator estimator: object, default=NoneThe base estimator from which the boosted ensemble is built.If ``None``, then the base estimator is:class:`~sklearn.tree.DecisionTreeRegressor` initialized with`max_depth=3`... versionadded:: 1.2 `base_estimator` was renamed to `estimator`.",DecisionTreeR...ndom_state=42)
,"n_estimators n_estimators: int, default=50The maximum number of estimators at which boosting is terminated.In case of perfect fit, the learning procedure is stopped early.Values must be in the range `[1, inf)`.",150
,"learning_rate learning_rate: float, default=1.0Weight applied to each regressor at each boosting iteration. A higherlearning rate increases the contribution of each regressor. There isa trade-off between the `learning_rate` and `n_estimators` parameters.Values must be in the range `(0.0, inf)`.",0.05
,"loss loss: {'linear', 'square', 'exponential'}, default='linear'The loss function to use when updating the weights after eachboosting iteration.",'square'
,"random_state random_state: int, RandomState instance or None, default=NoneControls the random seed given at each `estimator` at eachboosting iteration.Thus, it is only used when `estimator` exposes a `random_state`.In addition, it controls the bootstrap of the weights used to train the`estimator` at each boosting iteration.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
Name,Type,Value
estimator_ estimator_: estimatorThe base estimator from which the ensemble is grown... versionadded:: 1.2 `base_estimator_` was renamed to `estimator_`.,DecisionTreeRegressor,DecisionTreeR...ndom_state=42)
estimator_errors_ estimator_errors_: ndarray of floatsRegression error for each estimator in the boosted ensemble.,"ndarray[float64](150,)","[0.08,0.06,0.07,...,0.24,0.17,0.16]"
estimator_weights_ estimator_weights_: ndarray of floatsWeights for each estimator in the boosted ensemble.,"ndarray[float64](150,)","[0.12,0.14,0.13,...,0.06,0.08,0.08]"
estimators_ estimators_: list of regressorsThe collection of fitted sub-estimators.,list,"[DecisionTreeR...te=1608637542), DecisionTreeR...ate=448115643), DecisionTreeR...ate=250235941), DecisionTreeR...te=1734779151), ...]"
"feature_importances_ feature_importances_: ndarray of shape (n_features,)The impurity-based feature importances if supported by the``estimator`` (when based on decision trees).Warning: impurity-based feature importances can be misleading forhigh cardinality features (many unique values). See:func:`sklearn.inspection.permutation_importance` as an alternative.","ndarray[float64](49,)","[0.09,0. ,0. ,...,0. ,0. ,0. ]"


In [292]:
adaboost_predictions = adaboost_model.predict(X_validation)

In [293]:
adaboost_mae = mean_absolute_error(
    y_validation,
    adaboost_predictions
)

adaboost_rmse = np.sqrt(
    mean_squared_error(
        y_validation,
        adaboost_predictions
    )
)

adaboost_r2 = r2_score(
    y_validation,
    adaboost_predictions
)

print("AdaBoost Results")
print("MAE:", adaboost_mae)
print("RMSE:", adaboost_rmse)
print("R2:", adaboost_r2)

AdaBoost Results
MAE: 12.585077557967772
RMSE: 16.550736732460265
R2: 0.8425884988100056


In [294]:
comparison = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Random Forest",
        "Decision Tree",
        "SVR",
        "KNN",
        "Bagging + Decision Tree",
        "AdaBoost + Decision Tree"
    ],
    "MAE": [
        mae,
        rf_mae,
        dt_mae,
        svr_mae,
        knn_mae,
        bagging_mae,
        adaboost_mae
    ],
    "RMSE": [
        rmse,
        rf_rmse,
        dt_rmse,
        svr_rmse,
        knn_rmse,
        bagging_rmse,
        adaboost_rmse
    ],
    "R2": [
        r2,
        rf_r2,
        dt_r2,
        svr_r2,
        knn_r2,
        bagging_r2,
        adaboost_r2
    ]
})

comparison

,Model,MAE,RMSE,R2
0,Linear Regression,14.179876,17.492684,0.824161
1,Random Forest,10.035324,15.157266,0.867979
2,Decision Tree,12.989695,20.548327,0.757364
3,SVR,11.339444,15.572720,0.860642
4,KNN,11.557505,16.221334,0.848792
5,Bagging + Decision Tree,9.998103,15.139177,0.868294
6,AdaBoost + Decision Tree,12.585078,16.550737,0.842588


In [295]:
############### Test Data

In [296]:
test_df = pd.read_csv(
    "test_FD001.txt",
    sep=r"\s+",
    header=None
)

test_df.columns = columns

In [297]:
test_df = test_df.drop(
    columns=constant_columns
)

test_df = test_df.sort_values(
    by=["unit_id", "cycle"]
).reset_index(drop=True)

In [298]:
X_test_engineered = test_df.drop(
    columns=["unit_id"]
).copy()

X_test_engineered["cycle_squared"] = (
    X_test_engineered["cycle"] ** 2
)

In [299]:
test_sensor_columns = [
    col for col in test_df.columns
    if col.startswith("sensor_")
]

In [300]:
for sensor in test_sensor_columns:

    rolling_mean_values = (
        test_df.groupby("unit_id")[sensor]
        .rolling(
            window=5,
            min_periods=1
        )
        .mean()
        .reset_index(level=0, drop=True)
        .sort_index()
    )

    X_test_engineered[
        f"{sensor}_rolling_mean"
    ] = rolling_mean_values.values

In [301]:
for sensor in test_sensor_columns:

    rolling_std_values = (
        test_df.groupby("unit_id")[sensor]
        .rolling(
            window=5,
            min_periods=2
        )
        .std()
        .reset_index(level=0, drop=True)
        .sort_index()
    )

    X_test_engineered[
        f"{sensor}_rolling_std"
    ] = rolling_std_values.fillna(0).values

In [305]:
latest_indices = (
    test_df.groupby("unit_id")["cycle"].idxmax()
)

In [306]:
latest_indices = (
    test_df.loc[latest_indices]
    .sort_values("unit_id")
    .index
)

In [307]:
X_test_latest = X_test_engineered.loc[
    latest_indices
].reset_index(drop=True)

In [308]:
X_test_latest = X_test_latest.reindex(
    columns=X_train.columns
)

In [309]:
print("Test features shape:", X_test_latest.shape)
print("Expected features:", X_train.shape[1])
print("Number of test engines:", len(X_test_latest))
print("Missing values:", X_test_latest.isnull().sum().sum())

Test features shape: (100, 49)
Expected features: 49
Number of test engines: 100
Missing values: 0


In [ ]:
##### Final Training and Test Evaluation

In [310]:
final_model = bagging_model

final_model.fit(
    X_engineered,
    y
)

,"estimator estimator: object, default=NoneThe base estimator to fit on random subsets of the dataset.If None, then the base estimator is a:class:`~sklearn.tree.DecisionTreeRegressor`... versionadded:: 1.2 `base_estimator` was renamed to `estimator`.",DecisionTreeR...ndom_state=42)
,"n_estimators n_estimators: int, default=10The number of base estimators in the ensemble.",100
,"max_samples max_samples: int or float, default=NoneThe number of samples to draw from X to train each base estimator (withreplacement by default, see `bootstrap` for more details).- If None, then draw `X.shape[0]` samples irrespective of `sample_weight`.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` unweighted samples or `max_samples * sample_weight.sum()` weighted samples.",0.8
,"random_state random_state: int, RandomState instance or None, default=NoneControls the random resampling of the original dataset(sample wise and feature wise).If the base estimator accepts a `random_state` attribute, a differentseed is generated for each instance in the ensemble.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator (without replacement by default, see `bootstrap_features` for moredetails).- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.",1.0
,"bootstrap bootstrap: bool, default=TrueWhether samples are drawn with replacement. If False, sampling withoutreplacement is performed. If fitting with `sample_weight`, it isstrongly recommended to choose True, as only drawing with replacementwill ensure the expected frequency semantics of `sample_weight`.",True
,"bootstrap_features bootstrap_features: bool, default=FalseWhether features are drawn with replacement.",False
,"oob_score oob_score: bool, default=FalseWhether to use out-of-bag samples to estimatethe generalization error. Only available if bootstrap=True.",False
,"warm_start warm_start: bool, default=FalseWhen set to True, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fita whole new ensemble. See :term:`the Glossary <warm_start>`.",False
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for both :meth:`fit` and:meth:`predict`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"verbose verbose: int, default=0Controls the verbosity when fitting and predicting.",0


In [311]:
y_test = np.loadtxt(
    "RUL_FD001.txt"
)

print("True RUL shape:", y_test.shape)

True RUL shape: (100,)


In [ ]:
test_predictions = final_model.predict(
    X_test_latest
)

test_predictions = np.maximum(
    test_predictions,
    0
)

In [313]:
test_mae = mean_absolute_error(
    y_test,
    test_predictions
)

test_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        test_predictions
    )
)

test_r2 = r2_score(
    y_test,
    test_predictions
)

print("Final Test Results")
print("MAE:", test_mae)
print("RMSE:", test_rmse)
print("R2:", test_r2)

Final Test Results
MAE: 13.733437706100036
RMSE: 18.453004930130678
R2: 0.8028146474108832


In [314]:
y_test_capped = np.clip(
    y_test,
    a_min=None,
    a_max=125
)

In [315]:
test_capped_mae = mean_absolute_error(
    y_test_capped,
    test_predictions
)

test_capped_rmse = np.sqrt(
    mean_squared_error(
        y_test_capped,
        test_predictions
    )
)

test_capped_r2 = r2_score(
    y_test_capped,
    test_predictions
)

print("Capped Test Results")
print("MAE:", test_capped_mae)
print("RMSE:", test_capped_rmse)
print("R2:", test_capped_r2)

Capped Test Results
MAE: 12.663437706100035
RMSE: 17.314345895391295
R2: 0.8133179893202562


In [316]:
print("Test RUL values greater than 125:", (y_test > 125).sum())
print("Maximum test RUL:", y_test.max())

Test RUL values greater than 125: 11
Maximum test RUL: 145.0


In [ ]:
############# save final model 

In [319]:
model_artifact = {
    "model": final_model,
    "feature_columns": X_train.columns.tolist(),
    "constant_columns": constant_columns,
    "columns": columns,
    "rolling_window": 5
}

joblib.dump(
    model_artifact,
    "bagging_rul_model.pkl"
)

['bagging_rul_model.pkl']

In [322]:
print(os.path.exists("bagging_rul_model.pkl"))

True


In [303]:
tr = pd.read_csv('train_FD001.txt', sep=r'\s+', header=None)
profile = ProfileReport(tr, title="Profiling Report")

In [304]:
profile.to_file("nasa_report.html")


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 26/26 [00:00<00:00, 667.52it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]